# gg: full Soup Kernel Challenge validation
Enable Internet and GPU T4 x2. Add a read-only GITHUB_TOKEN Kaggle secret for this private repository.
Run All clones main. For final submission, pin COMMIT to the validated SHA and save a public version.
The source includes original kernels in solution/. T4 results remain pending until this run passes.


In [ ]:
REPO_URL = "https://github.com/vlvkindie/gg.git"
COMMIT = ""  # main; pin the validated SHA before final submission


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
subprocess.run(["python", "-m", "pip", "install", "-q", "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0", "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)


In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
import subprocess
repo = "/kaggle/working/repo"
subprocess.run(["python", "scripts/run_validation.py", "full"], cwd=repo, check=True)


In [ ]:
print("Results: /kaggle/working/repo/runs/result.json and runs/checks.json")


In [ ]:
subprocess.run(["python", "scripts/report.py"], cwd=repo, check=True)


While iterating, a quicker run of your solution alone (checks 1 and 5-8 need the base, so they fail
in this mode): `python harness/run_task.py --only solution --out runs_try`. Add `--profile` to get a
table of every CUDA kernel's GPU time in `runs/solution_profile/profile.json`. Logs of every run
are in `runs/<run>/train.log`.